# 06 · Reentrenamiento final con el 100 % de `train.csv`

**Contexto (16 de agosto de 2026, la víspera de la competencia).** El catedrático entregó dos
archivos de muestra —`data/raw/pipeline_test.csv` y `data/raw/expected_output.csv`— con la misma
estructura que tendrá el dataset de prueba real del lunes, para validar que el pipeline los
consume y devuelve el formato exacto que se espera.

Este notebook **no reabre ninguna decisión de modelado**: la arquitectura, la regularización y los
hiperparámetros quedan exactamente como en `data/processed/mejor_config.json` (iteración `it38`,
elegida y confirmada con 8 semillas en `04_iteraciones.ipynb` / `05_final.ipynb`). Lo único que
cambia es **cuántos datos ve el modelo que se usará mañana**:

- El modelo actual en `models/` se entrenó únicamente con `train_dev` (992 de 1168 filas, 85 %),
  porque el 15 % restante (`test interno`, 176 filas) se reservó para medir el RMSE una sola vez
  sin fuga de información.
- Esa medición ya se hizo y quedó documentada (`RMSE test interno = 24,197 USD`,
  `INFORME.md` §2.4). Ya no hace falta seguir reservando esas 176 filas: **la decisión sobre qué
  configuración usar ya está tomada**, así que usarlas ahora para entrenar no es fuga de
  información hacia una decisión — no hay ninguna decisión pendiente que puedan contaminar.
- Reentrenar con el 100 % (1168 filas) le da al modelo un 17.7 % más de datos, con la misma
  regularización (dropout 0.35, batch norm, weight decay 1e-2) que ya demostró necesaria contra el
  sobreajuste. Más datos con la misma capacidad y la misma regularización debería, si acaso,
  **reducir** el riesgo de sobreajuste, no aumentarlo.

**Qué NO se hizo:** no se usó `expected_output.csv` para validar precisión. Sus valores de
`Prediction` (1, 2, 3, 4, 5) son un marcador de posición para mostrar el formato de salida —no son
precios reales— así que no sirven para calcular RMSE. Solo confirman las columnas y el orden de
`Id` que debe tener la salida de `predict.py`.

Este notebook queda documentado como **Anexo** en `INFORME.md` y en la guía, sin modificar el
contenido ya entregado.

In [1]:
import json, sys, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path.cwd().parent))
from src.data import ROOT, load_raw, rmse, split_xy, ID_COL, TARGET
from src.model import ModelConfig, fit_mlp, predict_usd
from src.preprocessing import PreprocessConfig, build_preprocessor

MODELS = ROOT / "models"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


## 1. Cargar el dataset completo y la configuración ganadora (sin tocarla)

In [2]:
df = load_raw(ROOT / "data" / "raw" / "train.csv")
X, y = split_xy(df)
mejor = json.loads((ROOT / "data" / "processed" / "mejor_config.json").read_text())

CFG_PRE = PreprocessConfig(**mejor["pre"])
CFG_MLP = ModelConfig(**{**mejor["mlp"], "hidden": tuple(mejor["mlp"]["hidden"])})

print(f"Iteración ganadora : {mejor['id_iteracion']} — RMSE CV histórico {mejor['val_rmse_cv']:,.0f} USD")
print(f"Dataset completo    : {X.shape}")
print(f"Config preprocesamiento: {CFG_PRE.to_dict()}")
print(f"Config modelo          : {CFG_MLP.to_dict()}")

Iteración ganadora : it38 — RMSE CV histórico 28,375 USD
Dataset completo    : (1168, 79)
Config preprocesamiento: {'derived': True, 'log_skewed': True, 'skew_threshold': 0.75, 'drop_quasi_constant': True, 'rare_min_count': 0, 'scale': True}
Config modelo          : {'hidden': (128, 64), 'dropout': 0.35, 'batchnorm': True, 'activation': 'relu', 'lr': 0.001, 'weight_decay': 0.01, 'batch_size': 64, 'max_epochs': 600, 'patience': 80, 'scheduler': 'cosine', 'loss': 'mse', 'log_target': True, 'seed': 42}


## 2. Determinar la cantidad de épocas con un split de early stopping

Igual que en `05_final.ipynb`: se reserva un 12 % **solo para saber cuándo frenar**, no para
decidir nada del modelo (eso ya está decidido). La semilla es la misma (42) para no introducir una
fuente nueva de variación.

In [3]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X))
i_fit, i_es = train_test_split(idx, test_size=0.12, random_state=42)

pipeline_probe = build_preprocessor(CFG_PRE)
A_fit = pipeline_probe.fit_transform(X.iloc[i_fit], y.iloc[i_fit])
A_es = pipeline_probe.transform(X.iloc[i_es])
print(f"ajuste {A_fit.shape} | early stopping {A_es.shape}")

t0 = time.perf_counter()
modelo_probe, historia = fit_mlp(A_fit, y.iloc[i_fit].to_numpy(), A_es, y.iloc[i_es].to_numpy(),
                                  CFG_MLP, device=DEVICE)
print(f"mejor época: {historia.best_epoch} | RMSE early-stop: {historia.best_val_rmse:,.0f} USD "
      f"({time.perf_counter()-t0:.0f}s)")

ajuste (1027, 218) | early stopping (141, 218)


mejor época: 69 | RMSE early-stop: 22,806 USD (4s)


## 3. Reentrenar con el 100 % de los datos por el número de épocas encontrado

El pipeline se reajusta sobre las 1168 filas completas —no sobre `train_dev`— y el modelo se
entrena por el número fijo de épocas que encontró el paso anterior (sin más early stopping: ya se
sabe cuándo frenar, y frenar de nuevo sobre un split distinto solo agregaría varianza).

In [4]:
pipeline_final = build_preprocessor(CFG_PRE)
A_todo = pipeline_final.fit_transform(X, y)
A_es_final = pipeline_final.transform(X.iloc[i_es])
print(f"features: probe {A_fit.shape[1]} -> pipeline final {A_todo.shape[1]}")

epocas = historia.best_epoch + 1
cfg_final = ModelConfig(**{**CFG_MLP.to_dict(), "max_epochs": epocas, "patience": epocas})

t0 = time.perf_counter()
modelo_final, _ = fit_mlp(A_todo, y.to_numpy(), A_es_final, y.iloc[i_es].to_numpy(),
                           cfg_final, device=DEVICE)
modelo_final = modelo_final.cpu()
print(f"entrenado por {epocas} épocas ({time.perf_counter()-t0:.0f}s)")

features: probe 218 -> pipeline final 219


entrenado por 70 épocas (2s)


## 4. Chequeo de humo (NO es una métrica de validación)

Las 176 filas que antes eran el test interno ahora forman parte del entrenamiento, así que medir
el error sobre ellas estaría contaminado por fuga de información y daría un número optimista. Este
paso solo sirve para detectar **errores gruesos** (NaNs, escalas absurdas, signos invertidos) —no
se reporta como RMSE de generalización en ningún documento.

In [5]:
pred_todo = predict_usd(modelo_final, A_todo, cfg_final, device="cpu")
pred_todo = np.clip(pred_todo, 0, None)
rmse_optimista = rmse(y, pred_todo)

print("Chequeo de humo (con fuga, NO es una estimación de generalización):")
print(f"  RMSE sobre TODO el train (visto en entrenamiento): {rmse_optimista:,.0f} USD")
print(f"  Rango de predicciones: {pred_todo.min():,.0f} - {pred_todo.max():,.0f} USD")
print(f"  Rango real           : {y.min():,.0f} - {y.max():,.0f} USD")
assert np.isfinite(pred_todo).all(), "hay predicciones no finitas"
assert pred_todo.min() > 0, "hay predicciones negativas"
print("\nSin NaNs, sin negativos, en el rango esperado: OK")

Chequeo de humo (con fuga, NO es una estimación de generalización):
  RMSE sobre TODO el train (visto en entrenamiento): 14,311 USD
  Rango de predicciones: 38,856 - 561,842 USD
  Rango real           : 34,900 - 745,000 USD

Sin NaNs, sin negativos, en el rango esperado: OK


## 5. Guardar artefactos para la competencia

Sobrescribe `models/final_pipeline.joblib`, `models/final_model.pt` y `models/metadata.json`. La
versión anterior (entrenada con 992 filas) queda recuperable en el historial de git — no hace
falta un respaldo manual.

In [6]:
joblib.dump(pipeline_final, MODELS / "final_pipeline.joblib")
torch.save({
    "state_dict": modelo_final.state_dict(),
    "y_mu": modelo_final.y_mu_, "y_sigma": modelo_final.y_sigma_,
    "ensemble": None,
}, MODELS / "final_model.pt")

metadata_anterior = json.loads((MODELS / "metadata.json").read_text())

metadata = {
    "iteracion": mejor["id_iteracion"],
    "n_features": int(A_todo.shape[1]),
    "n_modelos": 1,
    "epocas_entrenamiento": epocas,
    "pre_config": CFG_PRE.to_dict(),
    "model_config": CFG_MLP.to_dict(),
    "excluir_ids": [],
    "rmse_cv_train_dev": float(mejor["val_rmse_cv"]),
    "rmse_test_interno_modelo_anterior_992": metadata_anterior.get("rmse_test_interno"),
    "n_train": int(len(X)),
    "n_train_modelo_anterior": metadata_anterior.get("n_train"),
    "reentrenado_full_dataset": True,
    "fecha_reentrenamiento": "2026-08-16",
    "motivo": ("Aprovechar el 100% de train.csv (1168 filas) para el modelo de competencia, "
               "sin reabrir ninguna decisión de arquitectura/hiperparametros (ya confirmadas con "
               "8 semillas en 04_iteraciones.ipynb). El RMSE de referencia para la competencia "
               "sigue siendo el de CV (28,375 +/- 892 USD), no un numero recalculado aqui, porque "
               "ya no queda un conjunto de test interno sin fuga de informacion."),
    "lectura_csv": "keep_default_na=False, na_values=['']",
}
(MODELS / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

print("Artefactos guardados en models/:")
for f in sorted(MODELS.glob("*")):
    if f.is_file():
        print(f"  {f.name:<24} {f.stat().st_size/1024:>8,.0f} KB")
print()
print(json.dumps(metadata, indent=2, ensure_ascii=False))

Artefactos guardados en models/:
  .gitkeep                        0 KB
  final_model.pt                151 KB
  final_pipeline.joblib          16 KB
  metadata.json                   1 KB

{
  "iteracion": "it38",
  "n_features": 219,
  "n_modelos": 1,
  "epocas_entrenamiento": 70,
  "pre_config": {
    "derived": true,
    "log_skewed": true,
    "skew_threshold": 0.75,
    "drop_quasi_constant": true,
    "rare_min_count": 0,
    "scale": true
  },
  "model_config": {
    "hidden": [
      128,
      64
    ],
    "dropout": 0.35,
    "batchnorm": true,
    "activation": "relu",
    "lr": 0.001,
    "weight_decay": 0.01,
    "batch_size": 64,
    "max_epochs": 600,
    "patience": 80,
    "scheduler": "cosine",
    "loss": "mse",
    "log_target": true,
    "seed": 42
  },
  "excluir_ids": [],
  "rmse_cv_train_dev": 28374.722099765862,
  "rmse_test_interno_modelo_anterior_992": 24197.4257202533,
  "n_train": 1168,
  "n_train_modelo_anterior": 992,
  "reentrenado_full_dataset": true,

## 6. Prueba de extremo a extremo con `predict.py` sobre el archivo de muestra del catedrático

Simula exactamente el flujo del lunes: correr `predict.py` sobre `pipeline_test.csv` (el archivo
de muestra, mismo formato que tendrá el dataset real) y comparar la forma de la salida —columnas y
orden de `Id`— contra `expected_output.csv`. Los valores de `expected_output.csv` son un
marcador de posición (1, 2, 3, 4, 5), no precios reales, así que esta prueba valida **formato**,
no precisión.

In [7]:
import subprocess

r = subprocess.run(
    [sys.executable, str(ROOT / "predict.py"),
     "--input", str(ROOT / "data" / "raw" / "pipeline_test.csv"),
     "--output", str(ROOT / "submissions" / "_pipeline_test_predicciones.csv")],
    capture_output=True, text=True, cwd=ROOT)
print(r.stdout or r.stderr)

pred = pd.read_csv(ROOT / "submissions" / "_pipeline_test_predicciones.csv")
esperado = pd.read_csv(ROOT / "data" / "raw" / "expected_output.csv")

print(pred)
print()
print(f"Columnas iguales           : {list(pred.columns) == list(esperado.columns)}")
print(f"Ids iguales y en el mismo orden: {list(pred['Id']) == list(esperado['Id'])}")
print(f"Filas: {len(pred)} predichas vs {len(esperado)} esperadas")

Predicciones : 5 filas → /home/escu/Documentos/Universidad/Semestres/8voSemestre/DEEP_LEARNING/pry1/submissions/_pipeline_test_predicciones.csv
Modelos      : 1 (promediados)
Rango        : 99,855 – 352,440 USD
Media        : 216,714 USD

(el CSV no trae SalePrice, así que no se puede calcular el RMSE)

     Id  Prediction
0   893  148767.500
1  1106  307910.750
2   414   99855.445
3   523  174596.420
4  1037  352439.600

Columnas iguales           : True
Ids iguales y en el mismo orden: True
Filas: 5 predichas vs 5 esperadas
